In [1]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:

from langchain_groq import ChatGroq

model=ChatGroq(model_name="llama-3.3-70b-versatile")
model



ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020065BC8830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020065BC9550>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
from langchain.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@tool
def subtract_numbers(a: int, b: int) -> int:
    """Subtract one number from another."""
    return a - b


model_with_tools=model.bind_tools([add_numbers, subtract_numbers])

# now i will invoke the model with a question that requires the use of the tool
response=model_with_tools.invoke("What is the sum of 5 and 7 and subtract 3?")
print(response)

# ab is response ko print kiya to you can see ki iske andr na ek tool_calls krke hai jo store krta hai ki LLM n kon si tool call ki hai..and us tool call me kya argument pass ki hui hai
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
   





content='' additional_kwargs={'tool_calls': [{'id': 'gaq4rqtrh', 'function': {'arguments': '{"a":5,"b":7}', 'name': 'add_numbers'}, 'type': 'function'}, {'id': 'zdw3p03ee', 'function': {'arguments': '{"a":12,"b":3}', 'name': 'subtract_numbers'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 290, 'total_tokens': 328, 'completion_time': 0.098171249, 'completion_tokens_details': None, 'prompt_time': 0.014626279, 'prompt_tokens_details': None, 'queue_time': 0.317366817, 'total_time': 0.112797528}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e84af-8a9a-7c40-b4a3-13e38f0cef7e-0' tool_calls=[{'name': 'add_numbers', 'args': {'a': 5, 'b': 7}, 'id': 'gaq4rqtrh', 'type': 'tool_call'}, {'name': 'subtract_numbers', 'args': {'a': 12, 'b': 3}, 'id': 'zdw3p03ee', 'type': 'tool_call'}] invalid_tool

In [14]:
### above this is happening
### You ask: "What is the sum of 5 and 7?"
### The LLM responds with a tool_calls entry saying: "call add_numbers with a=5, b=7"
### Missing step: You never actually invoke the tool function.

In [17]:
# To get the actual result, you need to execute the tool yourself:

# niche bata rkha hu why..
tools_available={"add_numbers": add_numbers, "subtract_numbers": subtract_numbers}
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")
    
    # Actually execute the tool

    # par ye to hardcoded ho gya ki add_number hi bula rhe
    # result = add_numbers.invoke(tool_call['args'])

    # chuki llm has already decided ki add_numbers tool ko call karna hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # and ye add_number to tool_call['name'] se match kar raha hai, to hum directly usko invoke kar dete hai with the arguments provided by the LLM
    # result=tool_call['name'].invoke(tool_call['args'])
    # but above line will give error kyuki tool_call['name'] is a string, not the actual function. So we need to map the string to the actual function.
    
    # isliye upar kahi pe hame add_numbers aur subtract_numbers ko define karna pada tha, taki hum unko map kar sakein with the string name of the tool.
    # like this: tools_available={"add_numbers": add_numbers, "subtract_numbers": subtract_numbers}

    # and phir jab hum let tool_call['Name'] me let "add_number" aya to phir ham tool_available["add_number"] se --add_number function invoke kar sakte hai with the arguments provided by the LLM.

    
    result = tools_available[tool_call['name']].invoke(tool_call['args'])

    # to you can see the output ki phela loop chala to add kiya and then substract kiya
    print(f"Result: {result}")

Tool called: add_numbers
Arguments: {'a': 5, 'b': 7}
Result: 12
Tool called: subtract_numbers
Arguments: {'a': 12, 'b': 3}
Result: 9
